# Test Notebook (using sample data)

To use the notebook download kernels and data: radiative kernels (spectral and broadband) are hosted separately on Zenodo
and Mendeley Data and are not included in the repository. 
Download them with:
```bash
cd spectfbcalc/
nohup python download_data.py > download_data.log 2>&1 &
disown
```
This runs in the background. Track progress with:
```bash
tail -f download_data.log
```
If interrupted, simply re-run the same command: already-downloaded files are skipped automatically.

## Import libraries and modules

In [ ]:
import spectfbcalc_lib as sfc
from climtools import climtools_lib as ctl 
import output_lib as out

In [ ]:
# test libraries import
sfc.mytestfunction()

In [ ]:
ctl.datestamp()

In [4]:
import sys
import os
import glob

import numpy as np
import xarray as xr

from matplotlib import pyplot as plt
import matplotlib.cbook as cbook

### OPTIONAL: launch workers to speed up the process (needs a SLURM scheduler)

In [ ]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

# dask will automatically submit SLURM jobs for you
cluster = SLURMCluster(
    cores=4,
    memory="64GB",
    processes=4,
    walltime="01:00:00",
    job_extra_directives=[
        "--account=spitfabi",
        "--qos=np"
    ]
)

# dask scale to desired number of workers
cluster.scale(jobs=4)  # This submits 4 SLURM jobs

# connect client
client = Client(cluster)

In [ ]:
print(client.dashboard_link)

In [ ]:
print(client)

In [ ]:
import dask.array as da
x = da.random.random((20000, 20000), chunks=(1000, 1000))
result = (x + x.T).mean().compute()
print(result)

# to check the status of the workers and the number of tasks executed, you can use the following code:
info = client.scheduler_info()['workers']
for addr, w in info.items():
    print(addr, "- tasks:", w.get('metrics', {}).get('task_counts', 'n/a'))

## Experiment setup

In [ ]:
config_file='config_template.yaml'
config = sfc.load_config(config_file, variable_mapping_file = None)

In [ ]:
config

In [ ]:
# load the control experiment object directly, data are already preprocessed and remapped.
ker = 'HUANG'  # or 'SPECTRAL'
raw_variables = {"hus", "rlut", "rsdt", "rlutcs", "rsus", "rsds", "rsut", "rsutcs", "ta", "tas", "ts"}
data_dir = config['file_paths']['reference_dataset']
cart_out = config['file_paths']['output']
control = sfc.Experiment('PI', orig_dir='', remap_dir = data_dir, raw_variables = raw_variables, variable_mapping = config['variable_mapping'], file_dict={})
experiment = sfc.Experiment('4x', orig_dir='', remap_dir=data_dir, raw_variables=raw_variables, variable_mapping=config['variable_mapping'], file_dict={})

control.load_remapped()
experiment.load_remapped()

In [ ]:
kernel = sfc.Kernel(ker, config=config)
k = kernel.kernel[('clr', 't')]

control.check_coords()
control.vertical_interp(k)
experiment.check_coords()
experiment.vertical_interp(k)

## Compute decomposed radiative anomalies

In [ ]:
# compute albedo
control.check_albedo()
experiment.check_albedo()

# check water vapor
control.check_vars('water-vapor', kernel.wv_name)
experiment.check_vars('water-vapor', kernel.wv_name)

# compute net TOA
control.compute_net_TOA()
experiment.compute_net_TOA()

In [ ]:
# compute climate anomalies (4x - PI)
sfc.compute_anomalies(experiment, control, method=config['anomaly_method'], time_range_clim=config['time_range_clim'])

In [ ]:
# compute anomalies one by one
sfc.Rad_anomaly_planck_surf(experiment, kernel, cart_out, save_pattern=True)

In [ ]:
sfc.Rad_anomaly_albedo(experiment, kernel, cart_out, save_pattern=True)

In [ ]:
sfc.Rad_anomaly_planck_atm_lr(experiment, kernel, cart_out, save_pattern=True)

In [ ]:
sfc.Rad_anomaly_wv(experiment, control, kernel, cart_out, save_pattern=True)

In [ ]:
sfc.Rad_anomaly_cloud(experiment, cart_out, save_pattern=True)

## Compute feedbacks

In [ ]:
# compute all feedbacks 
fb=sfc.calc_fb_from_exp(experiment, control, kernel, cart_out, save_pattern=config['save_pattern'], num_year_fb =config['num_year_regr'])

In [ ]:
# compute interannual feedbacks 
fb_interannual=sfc.calc_fb_interannual(experiment, control, kernel, cart_out)

## Save output and plot example

In [ ]:
import xarray as xr

out_dir = 'path/to/output/directory/'  # Replace with your desired output directory
out_path_txt = out_dir + 'res_fb.txt'
out_path_nc = out_dir + 'res_fb_patterns.nc'

# save both txt and NetCDF
out.save_feedback_output(fb, out_path_txt=out_path_txt, out_path_nc=out_path_nc)

In [ ]:
# gregory plot
feedback_file=out_path_txt
out.plot_feedback_slope(out_path_txt, sim_label="Test sample data")

In [ ]:
ds_patterns = xr.open_dataset(out_path_nc)

# plot all feedback patterns to a single PDF
out.save_all_fb_patterns_to_pdf(
    ds=ds_patterns, 
    output_folder=out_dir, 
    pdf_name="spatial_maps_feedback.pdf",
    run_label="Test sample data" 
)

In [ ]:
dRt_dict = sfc.open_dRt(cart_out, names=sfc.dRt_all + sfc.dRt_all_cloud)

# plot closure of the radiative budget at TOA, for both clear and all sky conditions
# clear sky
out.plot_toa_anomaly(
    experiment=experiment, 
    dRt_dict=dRt_dict, 
    title="Radiative Budget Closure - Clear Sky", 
    sky="clr", 
    output_file=out_dir + "budget_closure_clr.png"
)

# all sky
out.plot_toa_anomaly(
    experiment=experiment, 
    dRt_dict=dRt_dict, 
    title="Radiative Budget Closure - All Sky", 
    sky="cld", 
    output_file=out_dir + "budget_closure_cld.png"
)